In [1]:
#Import Libraries
import pandas as pd 
import sqlite3 
import json 

In [2]:
#Customer Data
customers = pd.DataFrame({
    "CustomerID": [ 
         "C001", "C002", "C003", "C004",
         "C004", "C005", "C006", "C007",
         "C008", "C009", "C010"
    ],
     "CustomerName": [
         "Amina", "Brian", "Carol", "David",
         "David", "Esther", "Faith", "George",
         "Halima", "Ian", "Jane" 
     ],
     "Segment": [
         "Retail", "Corporate", "Retail", "SME",
         "SME", "Corporate", "SME", "Retail",
         "Corporate", "Retail", "SME"
     ],
       "RegionCode": [
           "NRB", "MSA", "NRB", "KSM",
           "KSM", "NKR", "NRB", "MSA",
           "KSM", "NKR", "NRB"
       ]
}) 
customers

,CustomerID,CustomerName,Segment,RegionCode
0,C001,Amina,Retail,NRB
1,C002,Brian,Corporate,MSA
2,C003,Carol,Retail,NRB
3,C004,David,SME,KSM
4,C004,David,SME,KSM
5,C005,Esther,Corporate,NKR
6,C006,Faith,SME,NRB
7,C007,George,Retail,MSA
8,C008,Halima,Corporate,KSM
9,C009,Ian,Retail,NKR


In [3]:
customers.to_csv(
    "customers.csv",
    index=False
) 

In [4]:
#Orders Data
orders = pd.DataFrame({
    "OrderID": [
        "O001", "O002", "O003", "O004", "O005",
        "O006", "O007", "O008", "O009", "O010",
        "O011", "O012", "O013", "O014", "O015"
    ],
    "CustomerID": [
        "C001", "C002", "C001", "C003", "C004",
        "C005", "C006", "C002", "C007", "C008",
        "C009", "C010", "C006", "C003", "C999"
    ],
    "OrderAmount": [
         4500, 12500, 3200, 6800, 9100,
        15000, 7600, 8300, 2500, 11000,
        5400, 6200, 4300, 7200, 9900
    ],
    "ProductCategory": [
        "Electronics", "Office", "Home", "Electronics", "Office",
        "Electronics", "Home", "Office", "Home", "Electronics",
        "Office", "Home", "Electronics", "Office", "Electronics"
    ]
})
orders

,OrderID,CustomerID,OrderAmount,ProductCategory
0,O001,C001,4500,Electronics
1,O002,C002,12500,Office
2,O003,C001,3200,Home
3,O004,C003,6800,Electronics
4,O005,C004,9100,Office
5,O006,C005,15000,Electronics
6,O007,C006,7600,Home
7,O008,C002,8300,Office
8,O009,C007,2500,Home
9,O010,C008,11000,Electronics


In [5]:
orders.to_csv(
    "orders.csv",
    index=False
) 

In [6]:
#JSON Region Lookup
regions = [
    {"RegionCode": "NRB", "RegionName": "Nairobi"},
    {"RegionCode": "MSA", "RegionName": "Mombasa"},
    {"RegionCode": "KSM", "RegionName": "Kisumu"},
    {"RegionCode": "NKR", "RegionName": "Nakuru"}
]
with open("regions.json", "w") as file:
    json.dump(regions, file, indent=4)

In [7]:
#Inspecting the Sources
customers = pd.read_csv("customers.csv")
orders = pd.read_csv("orders.csv")

customers.head()
orders.head()

customers.shape
orders.shape 

(15, 4)

1. One row in customers.csv represents all information about a customer.
2. One row in orders.csv represents an order transaction.
3. The two datasets are not at the same aggregation grain.
This is because customer.csv is at the customer grain whereas orders.csv is at the orders grain.

Detecting Key Problems

In [10]:
#Duplicate Customer IDs
customers["CustomerID"].duplicated().sum()
customers[
  customers["CustomerID"].duplicated(keep=False)
]

,CustomerID,CustomerName,Segment,RegionCode
3,C004,David,SME,KSM
4,C004,David,SME,KSM


 4. A duplicate customer ID can be dangerous when joining customer and order tables because it can cause misleading results because one customer appears twice.

In [12]:
#Unmatched Customer IDs
set(orders["CustomerID"]) - set(customers["CustomerID"])

{'C999'}

5. The Customer ID that appears in orders but not in customers is C999.
6. An unmatched record should be investigated rather than silently deleted because investigating shows the root cause while deleting hides data quality issues.

In [14]:
#Creating the SQLite Database
conn = sqlite3.connect("retail_lab.db")
customers.to_sql(
    "customers",
    conn,
    if_exists="replace",
    index=False
)
orders.to_sql(
    "orders",
    conn,
    if_exists="replace",
    index=False
)

15

In [15]:
#Testing the Database
query = """
SELECT *
FROM customers
LIMIT 5;
"""

pd.read_sql_query(query, conn)

,CustomerID,CustomerName,Segment,RegionCode
0,C001,Amina,Retail,NRB
1,C002,Brian,Corporate,MSA
2,C003,Carol,Retail,NRB
3,C004,David,SME,KSM
4,C004,David,SME,KSM


TASK 1 - Basic SELECT

In [17]:
#A Query that Returns only OrderID, CustomerID and OrderAmount.
query = """
SELECT OrderID, CustomerID, OrderAmount
FROM orders;
"""  
pd.read_sql_query(query, conn)

,OrderID,CustomerID,OrderAmount
0,O001,C001,4500
1,O002,C002,12500
2,O003,C001,3200
3,O004,C003,6800
4,O005,C004,9100
5,O006,C005,15000
6,O007,C006,7600
7,O008,C002,8300
8,O009,C007,2500
9,O010,C008,11000


TASK 2 - Filter with WHERE

In [19]:
#Only customers in the Retail segment
query = """
SELECT *
FROM customers
WHERE Segment = 'Retail';
"""
pd.read_sql_query(query, conn)

,CustomerID,CustomerName,Segment,RegionCode
0,C001,Amina,Retail,NRB
1,C003,Carol,Retail,NRB
2,C007,George,Retail,MSA
3,C009,Ian,Retail,NKR


In [20]:
#Orders with OrderAmount greater than KSh 8,000
query = """
SELECT *
FROM orders
WHERE OrderAmount > 8000;
"""
pd.read_sql_query(query, conn)

,OrderID,CustomerID,OrderAmount,ProductCategory
0,O002,C002,12500,Office
1,O005,C004,9100,Office
2,O006,C005,15000,Electronics
3,O008,C002,8300,Office
4,O010,C008,11000,Electronics
5,O015,C999,9900,Electronics


In [21]:
#Electronics orders
query = """
SELECT *
FROM orders
WHERE ProductCategory = 'Electronics';
"""
pd.read_sql_query(query, conn)

,OrderID,CustomerID,OrderAmount,ProductCategory
0,O001,C001,4500,Electronics
1,O004,C003,6800,Electronics
2,O006,C005,15000,Electronics
3,O010,C008,11000,Electronics
4,O013,C006,4300,Electronics
5,O015,C999,9900,Electronics


In [22]:
#Electronics orders worth more than KSh 8,000
query = """
SELECT *
FROM orders
WHERE ProductCategory = 'Electronics'
AND OrderAmount > 8000;
"""
pd.read_sql_query(query, conn)

,OrderID,CustomerID,OrderAmount,ProductCategory
0,O006,C005,15000,Electronics
1,O010,C008,11000,Electronics
2,O015,C999,9900,Electronics


GROUP BY and Aggregate Functions

In [24]:
#Total Sales
query = """
SELECT SUM(OrderAmount) AS TotalSales
FROM orders;
"""
pd.read_sql_query(query, conn)

,TotalSales
0,113500


In [25]:
#Sales by Product Category
query = """
SELECT ProductCategory,
       SUM(OrderAmount) AS TotalSales
FROM orders
GROUP BY ProductCategory;
"""
pd.read_sql_query(query, conn)

,ProductCategory,TotalSales
0,Electronics,51500
1,Home,19500
2,Office,42500


In [26]:
#KPIs for each product category
query = """
SELECT ProductCategory,
       COUNT(*) AS NumberOfOrders,
       SUM(OrderAmount) AS TotalSales,
       AVG(OrderAmount) AS AverageOrderValue
FROM orders
GROUP BY ProductCategory;
"""
pd.read_sql_query(query, conn)

,ProductCategory,NumberOfOrders,TotalSales,AverageOrderValue
0,Electronics,6,51500,8583.333333
1,Home,4,19500,4875.000000
2,Office,5,42500,8500.000000


7. Observations about category performance:
- Home is the weakest performer having 4 NumberOfOrders, with TotalSales of 19500 and AverageOrderValue of 4875.00 while Electronics is the best performer having 6 NumberOfOrders, with TotalSales of 51500 and AverageOrderValue of 8583.33.
- Office has fewer orders than Electronics but has an AverageOrderValue of 8500 that is close to that of Electronics which shows that it has high-value transactions.

In [28]:
#JOIN Customers and Orders
query = """
SELECT
o.OrderID,
o.CustomerID,
c.CustomerName,
c.Segment,
o.OrderAmount
FROM orders o
LEFT JOIN customers c
ON o.CustomerID = c.CustomerID;
"""
joined = pd.read_sql_query(query, conn)
joined

,OrderID,CustomerID,CustomerName,Segment,OrderAmount
0,O001,C001,Amina,Retail,4500
1,O002,C002,Brian,Corporate,12500
2,O003,C001,Amina,Retail,3200
3,O004,C003,Carol,Retail,6800
4,O005,C004,David,SME,9100
5,O005,C004,David,SME,9100
6,O006,C005,Esther,Corporate,15000
7,O007,C006,Faith,SME,7600
8,O008,C002,Brian,Corporate,8300
9,O009,C007,George,Retail,2500


In [29]:
len(orders)
len(joined)

16

8. The row counts are not equal. This is because C004 has been duplicated.
9. The unmatched customer record in the joined output is C999. For CustomerName and Segment, it shows NULL values.

In [31]:
orders["OrderAmount"].sum()

113500

In [32]:
joined["OrderAmount"].sum()

122600

10. The two sales totals are not equal.
11. The customer that caused the duplication was David.
12. Total reconciliation is an important validation step after a JOIN because it catches errors before reporting wrong KPIs.

In [34]:
#Correct the Duplicate-Key Problem
customers_clean = (
    customers
    .drop_duplicates(subset="CustomerID")
)
customers_clean.to_sql(
    "customers_clean",
    conn,
    if_exists="replace",
    index=False
)
query = """
SELECT
   o.OrderID,
   o.CustomerID,
   c.CustomerName,
   c.Segment,
   o.OrderAmount
FROM orders o
LEFT JOIN customers_clean c
   ON o.CustomerID = c.CustomerID;
   """
clean_joined = pd.read_sql_query(query, conn)
clean_joined

,OrderID,CustomerID,CustomerName,Segment,OrderAmount
0,O001,C001,Amina,Retail,4500
1,O002,C002,Brian,Corporate,12500
2,O003,C001,Amina,Retail,3200
3,O004,C003,Carol,Retail,6800
4,O005,C004,David,SME,9100
5,O006,C005,Esther,Corporate,15000
6,O007,C006,Faith,SME,7600
7,O008,C002,Brian,Corporate,8300
8,O009,C007,George,Retail,2500
9,O010,C008,Halima,Corporate,11000


In [35]:
len(orders),
len(clean_joined)

15

In [36]:
orders["OrderAmount"].sum(),
clean_joined["OrderAmount"].sum()

113500

 13. The corrected JOIN has preserved both the row count and the total sales value. The number of rows is 15, not 16, and the total sales revenue is 113500, not 122600.

In [38]:
#Customer Segment KPIs
query = """
SELECT c.Segment,
       COUNT(o.OrderID) AS Orders,
       SUM(o.OrderAmount) AS TotalSales,
       AVG(o.OrderAmount) AS AverageOrderValue
FROM orders o
LEFT JOIN customers_clean c
ON o.CustomerID = c.CustomerID
GROUP BY c.Segment;
"""
pd.read_sql_query(query, conn)

,Segment,Orders,TotalSales,AverageOrderValue
0,None,1,9900,9900.000000
1,Corporate,4,46800,11700.000000
2,Retail,6,29600,4933.333333
3,SME,4,27200,6800.000000


14. From the segment KPIs, management can learn that:
-The NULL segment shows that the data has gaps that need fixing.
-Corporate customers contribute highly to TotalSales even with few orders.
-Retail has more orders but smaller average order values.

In [40]:
#Merge the JSON Region Lookup
regions_df = pd.read_json("regions.json")
regions_df 

,RegionCode,RegionName
0,NRB,Nairobi
1,MSA,Mombasa
2,KSM,Kisumu
3,NKR,Nakuru


In [41]:
customers_regions = customers_clean.merge(
    regions_df,
    on="RegionCode",
    how="left"
)
customers_regions.head()

,CustomerID,CustomerName,Segment,RegionCode,RegionName
0,C001,Amina,Retail,NRB,Nairobi
1,C002,Brian,Corporate,MSA,Mombasa
2,C003,Carol,Retail,NRB,Nairobi
3,C004,David,SME,KSM,Kisumu
4,C005,Esther,Corporate,NKR,Nakuru


15. Each customer now has a readable RegionName rather than only a RegionCode.
16. If a RegionCode did not exist in regions.json, a left merge would produce NULL (None) for RegionName.

In [43]:
# Save regions_df into SQLite
regions_df.to_sql(
    "regions",
    conn,
    if_exists="replace",
    index=False
)

4

In [44]:
#Regianal Sales Analysis
query = """
SELECT r.RegionName,
       COUNT(o.OrderID) AS NumberOfOrders,
       SUM(o.OrderAmount) AS TotalSales
FROM orders o
INNER JOIN customers_clean c ON o.CustomerID = c.CustomerID
INNER JOIN regions r ON c.RegionCode = r.RegionCode
GROUP BY r.RegionName;
"""
pd.read_sql_query(query, conn)

,RegionName,NumberOfOrders,TotalSales
0,Kisumu,2,20100
1,Mombasa,3,23300
2,Nairobi,7,39800
3,Nakuru,2,20400


17. Nairobi generated the highest sales.
18. Management should not automatically conclude that this is the “best-performing” region.
- It does not account for expenses or competition.
- It ignores profit margins.

- The biggest data-quality problem was duplicate CustomerID.
- It affected the JOIN by inflating totals.
- Totals should be checked after joining tables so as to realise errors.
- The customer segment that generated the most sales is Corporate.
- One limitation of the analysis was that it ignores profits.